# 02a — FLUX.1-dev Character Stills (pure diffusers, no ComfyUI)

Replaces the ComfyUI stills workflow (`02_stills` / `05_stills_character_lora`). Everything runs in
this notebook via `diffusers.FluxPipeline` — no headless server, no tunnel, no custom nodes.

**Identity** comes from your ai-toolkit FLUX LoRA (trained in `01c`). ai-toolkit saves it in a
diffusers-loadable key scheme (native PEFT `lora_A`/`lora_B`, or kohya `lora_down`/`lora_up` — the
diagnostic in §3 reports which), so `load_lora_weights` works out of the box. (Iteration 1's SDXL
LoRA was the *opposite* failure: a scheme ComfyUI couldn't read. This file is already in the shape
diffusers expects.)

**Guard rails baked in (lessons from iter-1):**
- Long downloads go to **log files**, never an unread PIPE.
- HF cache lives on **Drive** so the 24 GB FLUX download survives session resets.
- A **LoRA key diagnostic + A/B strength check** proves the LoRA actually loads before you trust it
  (the zero-effect bug that burned the SDXL LoRA). Never eyeball-only.

**VRAM:** bf16 FLUX.1-dev + T5-XXL + CLIP-L ≈ 24 GB resident. Strategy auto-selected from your GPU.
A100-80 runs everything on-GPU; A100-40/L4 uses group offloading or CPU offload.

**Prereqs:** a trained `Yuna_flux.safetensors` (or your character's) in
`Drive/.../ai_character_studio/loras/`, and `HF_TOKEN` in Colab Secrets with FLUX.1-dev access.

## 1. Config + mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'Yuna'
TRIGGER_TOKEN  = 'sks_vyuna'
LORA_STRENGTH  = 1.4     # validated on real gens 2026-09-06: likeness saturates ~1.4, 1.6 == 1.4
WIDTH, HEIGHT  = 1024, 1024
N_IMAGES       = 4       # how many per prompt
# ─────────────────────────────────────────────────────────────────────────

import os
DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
LORA_PATH  = f'{DRIVE_BASE}/loras/{CHARACTER_NAME}_flux_r64.safetensors'
IMG_OUT    = f'{DRIVE_BASE}/outputs/images/{CHARACTER_NAME}'
os.makedirs(IMG_OUT, exist_ok=True)

# Persist HF cache to Drive (FLUX.1-dev is ~24GB — don't re-download each session)
os.environ['HF_HOME'] = f'{DRIVE_BASE}/models/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

print(f'Character : {CHARACTER_NAME}  |  Trigger: {TRIGGER_TOKEN}')
print(f'LoRA      : {LORA_PATH}')
print('LoRA present:', os.path.exists(LORA_PATH))
print(f'Outputs   : {IMG_OUT}')

## 2. HuggingFace login + install (uv)
FLUX.1-dev is gated — needs the license accepted at
https://huggingface.co/black-forest-labs/FLUX.1-dev and a read token (Colab Secrets → `HF_TOKEN`).
Inference reuses Colab's own torch (no isolated venv — that was only needed for *training*). We use
`uv pip --system` for speed per the project revisit note.

In [ ]:
import os
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception:
    from getpass import getpass
    hf_token = getpass('Paste your HuggingFace token (needs FLUX.1-dev access): ')
# Some Colab builds return userdata.get() as a dict instead of a plain str
# (observed 2026-09-06 post-reload) -> os.environ assignment then raises
# "str expected, not dict". Normalize before use.
if isinstance(hf_token, dict):
    hf_token = hf_token.get('value') or next(iter(hf_token.values()), None)
if not isinstance(hf_token, str):
    raise RuntimeError(f'HF_TOKEN is not a string (got {type(hf_token).__name__}): {str(hf_token)[:80]!r}')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

# fast installer, reuse the pre-installed Colab torch (do NOT let uv reinstall torch)
!pip install -q uv

# diffusers stack. Pin to a recent release known to have FLUX + FluxPriorRedux + FluxKontext.
# --system → installs into Colab's own env so torch is shared.
!uv pip install --system -q --reinstall-package diffusers \
    "diffusers>=0.32.0" transformers accelerate peft bitsandbytes safetensors huggingface_hub

import torch, diffusers, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('diffusers', diffusers.__version__, '| transformers', transformers.__version__)

# Confirm FLUX.1-dev access before we commit to a 24GB download
from huggingface_hub import HfApi
try:
    HfApi().model_info('black-forest-labs/FLUX.1-dev', token=hf_token)
    print('✅ FLUX.1-dev access confirmed.')
except Exception as e:
    print('❌ No FLUX.1-dev access. Accept the license, then rerun this cell.')
    print('  ', str(e)[:200])

# torchao: Colab ships 0.10.0 but the peft uv pulls requires >=0.16.0 and its
# is_torchao_available() check raises instead of returning False. We don't use
# torchao (quantization lib) for bf16 FLUX/Wan/LTX + LoRA, so remove it to keep
# peft's LoRA injection path clean.
!uv pip uninstall --system -y torchao 2>/dev/null || true
# Newer diffusers (>=0.33 dev, e.g. 0.39.0.dev0) EAGERLY import
# diffusers.quantizers.torchao, whose guard is is_torchao_available() =
# find_spec("torchao"). If torchao is only HALF-removed (a stale .dist-info
# survives `uv pip uninstall`), the guard returns True but `import torchao`
# fails -> "No module named 'torchao'" at `from diffusers import FluxPipeline`.
# So rm the leftovers too, forcing the guard to False so diffusers skips torchao.
# We don't use torchao (a quantization lib) for bf16 FLUX/Wan/LTX + LoRA.
!rm -rf /usr/local/lib/python3.13/dist-packages/torchao \
       /usr/local/lib/python3.13/dist-packages/torchao-*.dist-info
# ── CUDA-torch guard ────────────────────────────────────────────────────────
# uv can silently re-resolve deps and swap Colab's CUDA torch for the PyPI CPU
# wheel ("Torch not compiled with CUDA enabled" at load time). Detect + repair
# here, BEFORE we commit to a multi-GB model download.
import torch as _t
print('torch', _t.__version__, '| cuda', _t.cuda.is_available())
if not _t.cuda.is_available():
    import subprocess
    ver = _t.__version__.split('+')[0]
    print(f'CPU-only torch {ver} detected (uv swapped it). Reinstalling the CUDA build from cu124...')
    subprocess.run(f'uv pip install --system "torch=={ver}" torchvision '
                   '--index-url https://download.pytorch.org/whl/cu124',
                   shell=True, check=True)
    raise RuntimeError(
        'CUDA torch reinstalled on disk. Now: Runtime > Restart runtime, then re-run '
        'this install cell + the model-load cell. (The running kernel still holds the '
        'old CPU torch in memory, so the restart is required — do not skip it.)')


## 3. LoRA key diagnostic (do this BEFORE trusting any LoRA)
Prints the top-level key prefixes and counts. For a valid ai-toolkit FLUX LoRA you should see
`lora_transformer_`, `lora_te1_`, and `lora_te2_` prefixes with `down`/`up` (or `A`/`B`) in the names.
If you only see `unet.`/`lora_A`-style PEFT keys with no `lora_*_` prefixing, the file is in a format
diffusers can't map to FLUX — stop and re-export (this is exactly the iter-1 failure, inverted).

In [ ]:
from safetensors import safe_open
import collections

with safe_open(LORA_PATH, framework='pt') as f:
    keys = list(f.keys())

print(f'{len(keys)} tensors in {LORA_PATH}\n')

# Which component does each key target? (format-agnostic — strip common prefixes
# so transformer vs text-encoder can be told apart no matter the naming family)
def component(k):
    kk = k
    for pref in ('lora_', 'base_model.model.', 'diffusion_model.'):
        if kk.startswith(pref):
            kk = kk[len(pref):]
    if kk.startswith('transformer.') or kk.startswith('transformer_'):
        return 'transformer'
    if (kk.startswith('text_models.') or kk.startswith('text_encoder')
            or kk.startswith('te1') or kk.startswith('te2')
            or kk.startswith('text_encoders.')):
        return 'text-encoder'
    return 'other'

counts = collections.Counter(component(k) for k in keys)
print('components:')
for name in ('transformer', 'text-encoder', 'other'):
    if counts.get(name):
        print(f'  {name}: {counts[name]}')

# Which LoRA param naming family? (native PEFT vs kohya — both load on FLUX)
def naming(k):
    if k.endswith('.lora_A.weight') or k.endswith('.lora_B.weight') or '.lora_A.' in k or '.lora_B.' in k:
        return 'PEFT lora_A/lora_B'
    if 'lora_down' in k or 'lora_up' in k:
        return 'kohya lora_down/lora_up'
    return 'unknown'

fam = collections.Counter(naming(k) for k in keys)
print('\nLoRA param naming:')
for name, n in fam.items():
    print(f'  {name}: {n}')

print('\nSample keys:')
for k in keys[:5]:
    print('  ', k)

has_transformer = counts.get('transformer', 0) > 0
has_text = counts.get('text-encoder', 0) > 0
loadable = fam.get('PEFT lora_A/lora_B', 0) > 0 or fam.get('kohya lora_down/lora_up', 0) > 0

if has_transformer and loadable:
    print('\n✅ FLUX LoRA format recognized (transformer keys in a diffusers-loadable scheme). Proceed.')
    if not has_text:
        print('   note: no text-encoder LoRA keys — trigger may bind weakly. Fine only if you trained transformer-only.')
else:
    print('\n❌ No recognized transformer LoRA keys, or an unfamiliar key scheme. '
          'This may not load on a FLUX pipeline — inspect the sample keys / full list before proceeding.')

with open(f'{IMG_OUT}/_lora_keys.txt', 'w') as fh:
    fh.write('\n'.join(keys))
print(f'Full key list -> {IMG_OUT}/_lora_keys.txt')


## 4. Auto-pick a VRAM strategy, then load FLUX.1-dev

In [ ]:
import torch
from diffusers import FluxPipeline

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {torch.cuda.get_device_name(0)}  ~{vram_gb:.0f} GB')

# Strategy tiers (all present so you can edit):
if vram_gb >= 60:
    STRAT = 'gpu'          # A100-80: everything resident
elif vram_gb >= 30:
    STRAT = 'group'        # A100-40: group offloading (fast, low VRAM)
else:
    STRAT = 'cpu'          # L4/24GB: sequential CPU offload (slow but fits)
print(f'Strategy: {STRAT}')

MODEL_ID = 'black-forest-labs/FLUX.1-dev'
dtype = torch.bfloat16
LOG = '/content/flux_load.log'

import logging
logging.basicConfig(filename=LOG, level=logging.INFO)

pipe = FluxPipeline.from_pretrained(MODEL_ID, dtype=dtype, token=os.environ['HF_TOKEN'])

if STRAT == 'gpu':
    pipe.to('cuda')
elif STRAT == 'group':
    from diffusers.hooks import apply_group_offloading
    onload, offload = torch.device('cuda'), torch.device('cpu')
    for comp in (pipe.transformer, pipe.text_encoder, pipe.text_encoder_2, pipe.vae):
        apply_group_offloading(comp, offload_device=offload, onload_device=onload,
                               offload_type='leaf_level', use_stream=True)
else:
    pipe.enable_sequential_cpu_offload()

# Always-on memory saviors, harmless at high VRAM
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

print('✅ FLUX.1-dev loaded.  Log →', LOG)

## 3. LoRA key diagnostic (do this BEFORE trusting any LoRA)
Reports which **components** the LoRA targets (transformer vs text-encoder) and which **param-naming
family** it uses (native PEFT `lora_A`/`lora_B` vs kohya `lora_down`/`lora_up`) — all diffusers-loadable
on a FLUX pipeline. A clean ✅ means the file has FLUX transformer keys in a recognized scheme; a ❌
means it looks unrecognizable, so inspect before spending a 24 GB load on it. This is a format screen,
not a behavior test — the real gate that the LoRA actually *changes* output is the A/B strength proof in
§5. The full key list is dumped to Drive (`_lora_keys.txt`) for offline inspection.


In [ ]:
import torch, os, hashlib
from PIL import Image

def png_sha(img):
    from io import BytesIO
    b = BytesIO(); img.save(b, 'PNG'); return hashlib.sha256(b.getvalue()).hexdigest()[:12]

# Version-robust strength setter: diffusers >=0.32 uses set_adapters(names, weights),
# older builds use set_adapters([weight]). Try new form, fall back to old.
def set_strength(pipe, s):
    # adapter name is 'default_0' (what load_lora_weights registers);
    # diffusers >=0.32: set_adapters(names, weights). Fallback: older one-arg form.
    try:
        pipe.set_adapters(['default_0'], [s])
    except Exception:
        pipe.set_adapters([s])

pipe.load_lora_weights(LORA_PATH)
# Explicitly set the active adapter weight
try:
    set_strength(pipe, LORA_STRENGTH)
    print(f'LoRA loaded, adapter set to {LORA_STRENGTH}.')
except Exception as e:
    print('set_adapters fallback:', e)

def gen(prompt, seed=1234, w=WIDTH, h=HEIGHT, steps=28):
    g = torch.Generator(device='cuda' if STRAT != 'cpu' else 'cpu').manual_seed(seed)
    return pipe(prompt=prompt, width=w, height=h,
                num_inference_steps=steps, guidance_scale=3.5,
                generator=g, max_sequence_length=512).images[0]

# --- A/B proof: same prompt + seed, two strengths ---------------------------
PROMPT = f'{TRIGGER_TOKEN}, portrait photo, detailed face, sharp eyes, natural skin texture, soft window light'

set_strength(pipe, 0.5)
img_a = gen(PROMPT, seed=1234)
set_strength(pipe, 1.0)
img_b = gen(PROMPT, seed=1234)

ha, hb = png_sha(img_a), png_sha(img_b)
print(f'strength 0.5 sha: {ha}')
print(f'strength 1.0 sha: {hb}')
if ha == hb:
    print('❌ IDENTICAL at 0.5 vs 1.0 → LoRA is NOT affecting output. Stop here — the keys '
          "didn't bind. Check the diagnostic in cell 3.")
else:
    print('✅ LoRA is active (0.5 ≠ 1.0). Restoring default strength.')
set_strength(pipe, LORA_STRENGTH)

# side-by-side sanity view
combo = Image.new('RGB', (img_a.width*2 + 10, img_a.height), 'black')
combo.paste(img_a, (0,0)); combo.paste(img_b, (img_a.width+10,0))
combo.save(f'{IMG_OUT}/_lora_ab_check.png')
from IPython.display import display
display(combo)
print('left = 0.5   right = 1.0   (should differ; both should look like the character)')


## 6. Generate stills
The core `generate_stills()` helper. Prompt should NOT re-describe the face/identity — the trigger
token + LoRA carry that. Describe pose, clothing, lighting, setting, composition.

In [ ]:
import torch, os, time
from pathlib import Path

def generate_stills(prompt, n=N_IMAGES, width=WIDTH, height=HEIGHT,
                    seed=-1, steps=28, guidance=3.5, ref_image=None, ip_scale=None):
    """
    Generate `n` stills of the character for `prompt`.
    - ref_image: PIL image or path → enables IP-Adapter face pull (only if loaded, §8).
    - ip_scale : IP-Adapter scale 0-1 (ignored unless an IP-Adapter is loaded).
    Returns list of saved PNG paths.
    """
    if not prompt.startswith(TRIGGER_TOKEN):
        prompt = f'{TRIGGER_TOKEN}, {prompt}'

    out_dir = Path(IMG_OUT) / time.strftime('%Y%m%d_%H%M%S')
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for i in range(n):
        s = seed if seed != -1 else (1000 + i * 97)
        g = torch.Generator(device='cuda' if STRAT != 'cpu' else 'cpu').manual_seed(s)
        kw = dict(prompt=prompt, width=width, height=height, num_inference_steps=steps,
                  guidance_scale=guidance, generator=g, max_sequence_length=512)
        if ref_image is not None and hasattr(pipe, 'set_ip_adapter_scale'):
            from diffusers.utils import load_image
            img = load_image(ref_image) if isinstance(ref_image, (str, Path)) else ref_image
            kw['ip_adapter_image'] = img
            if ip_scale is not None:
                pipe.set_ip_adapter_scale(ip_scale)
        im = pipe(**kw).images[0]
        p = out_dir / f'{i:02d}_s{s}.png'
        im.save(p)
        paths.append(str(p))
        print(f'  saved {p} (seed {s})')
    print(f'✅ {len(paths)} stills → {out_dir}')
    return paths

# ── First real generation: a few test shots ───────────────────────────────
shots = [
    'portrait, looking at camera, neutral expression, studio lighting, gray backdrop',
    'full body, standing, city street at dusk, cinematic, shallow depth of field',
    'close-up, slight smile, golden hour rim light, bokeh background',
    'upper body, arms crossed, overcast day, candid street style',
]
for s in shots:
    generate_stills(s, n=1)

## 7. Review grid

In [ ]:
from PIL import Image
from IPython.display import display, HTML
import base64, glob

def thumb(path, size=280):
    im = Image.open(path).convert('RGB'); im.thumbnail((size, size))
    from io import BytesIO
    b = BytesIO(); im.save(b, 'JPEG', quality=85)
    return f'<img src="data:image/jpeg;base64,{base64.b64encode(b.getvalue()).decode()}">'

latest = sorted(glob.glob(f'{IMG_OUT}/*/*.png'))[-12:]
html = '<h3>Latest stills</h3><div style="display:flex;flex-wrap:wrap;gap:8px">' + \
       ''.join(thumb(p) for p in latest) + '</div>'
display(HTML(html))

## 8. OPTIONAL — IP-Adapter for extra face pull (XLabs flux-ip-adapter)
The LoRA alone should hold identity. If the face drifts on tricky prompts, load the CLIP-based
IP-Adapter and pass a reference image. `~100 MB` extra download; needs `ip_adapter_image=` in
`generate_stills` (already wired). Keep scale 0.5–0.8 — too high and the reference's background/clothing
bleeds in.

**Other face-identity routes to try if IP-Adapter isn't enough (not loaded by default):**
- **Flux Redux** (`FluxPriorReduxPipeline`) — image-conditioned *composition* control, ~17 GB extra.
  Better for "put this exact scene/pose with my character" than raw face similarity. See §9.
- **Flux Kontext** (`FluxKontextPipeline`) — in-context editing/reference; strong character consistency,
  but a separate 12B model. See §9.
- **InsightFace/FaceID-for-FLUX** — ComfyUI had FaceID; the clean diffusers path is the IP-Adapter above.
  A face-crop **img2img detail pass** (re-denoise only the face region at low strength) is in §9.
- **ADetailer** — ComfyUI-only (Impact Pack). No direct diffusers equivalent; approximate with the
  face img2img pass in §9 if eyes come out soft.

In [ ]:
# Uncomment to enable IP-Adapter face pull. Needs the cell-4 `pipe` still loaded.
USE_IPADAPTER = False

if USE_IPADAPTER:
    # one-time ~100MB download (into the Drive HF cache from cell 2)
    pipe.load_ip_adapter(
        "XLabs-AI/flux-ip-adapter",
        weight_name="ip_adapter.safetensors",
        image_encoder_pretrained_model_name_or_path="openai/clip-vit-large-patch14",
    )
    pipe.set_ip_adapter_scale(0.6)
    print('✅ flux-ip-adapter loaded. Now call generate_stills(..., ref_image="path/ref.png").')

    # Reference = a clean front-facing still of the character (best: one of your training refs)
    REF = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}/reference-images/ref01.jpg'  # ← adjust
    generate_stills('upper body, hands in pockets, overcast day, candid', n=1,
                    ref_image=REF, ip_scale=0.6)

## 9. OPTIONAL — bigger/better stills routes (commented, try later)
These are the heavier models we'd like to A/B against the base FLUX.1-dev + LoRA pipeline. None are
active by default — pick one when you have the GPU time and the extra download budget.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# A) FLUX.1-Redux-dev  — image-conditioned COMPOSITION control (~17GB extra)
#    Good for "this exact scene/pose, but with my character".
#    Run AFTER the base pipe so it can share nothing (separate pipeline objects).
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import FluxPriorReduxPipeline, FluxPipeline
# from diffusers.utils import load_image
# pipe_prior_redux = FluxPriorReduxPipeline.from_pretrained(
#     "black-forest-labs/FLUX.1-Redux-dev", dtype=torch.bfloat16, token=os.environ['HF_TOKEN']
# ).to('cuda')
# pipe_redux = FluxPipeline.from_pretrained(
#     "black-forest-labs/FLUX.1-dev", text_encoder=None, text_encoder_2=None,
#     dtype=torch.bfloat16, token=os.environ['HF_TOKEN']).to('cuda')
# pipe_redux.load_lora_weights(LORA_PATH)
# try: pipe_redux.set_adapters(['default_0'], [LORA_STRENGTH])
# except Exception: pipe_redux.set_adapters([LORA_STRENGTH])
# ref = load_image('path/to/pose_or_scene_reference.png').convert('RGB')
# prior = pipe_prior_redux(ref)
# imgs = pipe_redux(guidance_scale=2.5, num_inference_steps=40, **prior).images
# imgs[0].save(f'{IMG_OUT}/redux.png')

# ─────────────────────────────────────────────────────────────────────────
# B) FLUX.1-Kontext-dev — in-context reference / editing (separate 12B model)
#    Very strong for "keep the same character, change the scene".
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import FluxKontextPipeline
# from diffusers.utils import load_image
# kpipe = FluxKontextPipeline.from_pretrained(
#     "black-forest-labs/FLUX.1-Kontext-dev", dtype=torch.bfloat16, token=os.environ['HF_TOKEN']
# )
# kpipe.to('cuda')
# base_img = load_image('path/to/a_good_character_still.png').convert('RGB')
# out = kpipe(image=base_img, prompt=f'{TRIGGER_TOKEN}, now standing in a rain-soaked alley at night',
#             guidance_scale=2.5).images[0]
# out.save(f'{IMG_OUT}/kontext.png')

# ─────────────────────────────────────────────────────────────────────────
# C) FLUX.1-schnell — 4-step FAST iteration (great for quick prompt sweeps)
#    NOT guidance-distilled → guidance_scale=0, max_sequence_length<=256.
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import FluxPipeline
# spipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell",
#     dtype=torch.bfloat16, token=os.environ['HF_TOKEN']); spipe.enable_model_cpu_offload()
# out = spipe(prompt=f'{TRIGGER_TOKEN}, portrait', guidance_scale=0., width=1024, height=1024,
#             num_inference_steps=4, max_sequence_length=256).images[0]

# ─────────────────────────────────────────────────────────────────────────
# D) Face/img2img DETAIL pass — approximate ADetailer: re-denoise the whole image at very low
#    strength to sharpen eyes/skin without changing composition. Crude but ComfyUI-free.
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import FluxImg2ImgPipeline
# from diffusers.utils import load_image
# fimg2img = FluxImg2ImgPipeline.from_pipe(pipe)  # reuses loaded FLUX.1-dev + LoRA
# src = load_image('path/to/soft_eyes_still.png').convert('RGB')
# sharp = fimg2img(image=src, prompt=f'{TRIGGER_TOKEN}, sharp detailed eyes, crisp skin texture',
#                  strength=0.25, num_inference_steps=12, guidance_scale=3.5).images[0]
# sharp.save(f'{IMG_OUT}/face_pass.png')

# ─────────────────────────────────────────────────────────────────────────
# E) FLUX.1-Fill-dev — inpainting (fix hands, blemishes, remove artifacts)
# ─────────────────────────────────────────────────────────────────────────
# from diffusers import FluxFillPipeline
# from diffusers.utils import load_image
# fill = FluxFillPipeline.from_pretrained("black-forest-labs/FLUX.1-Fill-dev",
#     dtype=torch.bfloat16, token=os.environ['HF_TOKEN']).to('cuda')
# out = fill(prompt='a natural hand', image=load_image('still.png'),
#            mask_image=load_image('hand_mask.png'), height=1024, width=1024).images[0]

# ─────────────────────────────────────────────────────────────────────────
# F) FP8 via optimum-quanto — run FLUX.1-dev under ~16GB VRAM (quality trade-off)
# ─────────────────────────────────────────────────────────────────────────
# !uv pip install --system -q optimum-quanto
# from optimum.quanto import freeze, qfloat8, quantize
# from diffusers import FluxPipeline, FluxTransformer2DModel
# from transformers import T5EncoderModel
# t2 = T5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder_2", dtype=torch.bfloat16)
# quantize(t2, weights=qfloat8); freeze(t2)
# pipe_fp8 = FluxPipeline.from_pretrained(MODEL_ID, text_encoder_2=None, dtype=torch.bfloat16)
# pipe_fp8.text_encoder_2 = t2
# pipe_fp8.transformer = FluxTransformer2DModel.from_pretrained(MODEL_ID, subfolder="transformer", dtype=torch.bfloat16)
# quantize(pipe_fp8.transformer, weights=qfloat8); freeze(pipe_fp8.transformer)
# pipe_fp8.enable_model_cpu_offload()

print('Section 9: all optional heavier routes are commented out. Enable one at a time.')

## 10. Log generation to metadata (library convention)

In [ ]:
import json, os, glob, time
meta_path = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}/metadata.json'
os.makedirs(os.path.dirname(meta_path), exist_ok=True)
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {'name': CHARACTER_NAME}

new_imgs = glob.glob(f'{IMG_OUT}/*/*.png')
meta.setdefault('stills_log', []).append({
    'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'base': 'flux-dev',
    'lora': os.path.basename(LORA_PATH),
    'strength': LORA_STRENGTH,
    'count': len(new_imgs),
    'paths': new_imgs[-10:],
})
json.dump(meta, open(meta_path, 'w'), indent=2)
print(f'metadata.json updated — {len(new_imgs)} stills logged.  → {meta_path}')
print('\n✅ 02a complete. These stills are the keyframe source for 03a / 03b (video Mode 1).')